# Pronunciation model — Kaggle GPU

1. **Add Data:** `tnguynthnh142/speechocean762`
2. **Settings:** GPU T4 + Internet ON
3. **Run All** → download `best_model.pt`

_OOM? batch_size=2 + grad_accum=4 (effective 8). WavLM frozen in inference_mode._


In [1]:
!pip install -q transformers torchaudio torch-geometric peft tqdm


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 24.9 MB/s eta 0:00:00


In [2]:
CONFIG = {
  "wavlm": {"model_name": "microsoft/wavlm-large", "freeze": True, "use_lora": False},
  "transformer": {"num_layers": 3, "num_heads": 8, "ff_dim": 2048, "dropout": 0.1, "max_seq_len": 800},
  "ctc_align": {"use_wavlm_ctc_head": True},
  "phoneme_graph": {"hidden_dim": 256, "num_gat_layers": 2, "num_heads": 4, "dropout": 0.1,
    "edge_types": {"sequential": True, "same_word": True, "same_syllable": False}},
  "multitask": {"hidden_dim": 256, "dropout": 0.1, "score_scale": 2.0,
    "utterance_aspects": ["accuracy","fluency","completeness","prosodic","total"],
    "word_aspects": ["accuracy","stress","total"], "phoneme_aspects": ["accuracy"]},
  "paths": {"checkpoint_dir": "/kaggle/working/checkpoints"},
  "train": {"batch_size": 2, "grad_accum_steps": 4, "num_epochs": 30, "learning_rate": 1e-4, "weight_decay": 0.01,
    "grad_clip": 1.0, "num_workers": 0, "pin_memory": True, "use_amp": True, "save_every_epochs": 5,
    "loss_weights": {"utterance": 1.0, "word": 1.0, "phoneme": 1.0},
    "dataset": {"data_dir": None, "train_split": "train", "test_split": "test",
      "max_duration_sec": 8.0, "sample_rate": 16000}},
}

import json
import os
from pathlib import Path
from typing import List, Optional, Tuple

# Kaggle mount (new UI): /kaggle/input/datasets/<user>/<slug>/...
KAGGLE_DATA_ROOT = Path("/kaggle/input/datasets/tnguynthnh142/speechocean762/speechocean762")

FALLBACK_ROOTS = (
    KAGGLE_DATA_ROOT,
    Path("/kaggle/input/speechocean762/speechocean762"),
    Path("/kaggle/input/speechocean762"),
)

def is_speechocean762_root(root: Path) -> bool:
    root = Path(root)
    return (
        (root / "train" / "wav.scp").is_file()
        and (root / "test" / "wav.scp").is_file()
        and ((root / "resource" / "scores.json").is_file() or (root / "scores.json").is_file())
    )

def find_speechocean762_dir(input_root: Optional[str] = None) -> Path:
    override = os.environ.get("KAGGLE_DATA_DIR")
    if override:
        root = Path(override)
        if is_speechocean762_root(root):
            return root
        raise FileNotFoundError(f"Invalid KAGGLE_DATA_DIR: {root}")

    for p in FALLBACK_ROOTS:
        if is_speechocean762_root(p):
            return p

    base = Path(input_root or "/kaggle/input")
    if base.is_dir():
        for wav_scp in base.rglob("wav.scp"):
            if wav_scp.parent.name == "train":
                root = wav_scp.parent.parent
                if is_speechocean762_root(root):
                    return root

    raise FileNotFoundError(
        "Dataset not found. Add Data: tnguynthnh142/speechocean762\n"
        f"Expected e.g. {KAGGLE_DATA_ROOT}"
    )

def verify_speechocean762(root: Path) -> Tuple[bool, List[str], dict]:
    root = Path(root)
    issues: List[str] = []
    info = {"root": str(root)}

    if not is_speechocean762_root(root):
        return False, ["Invalid dataset root"], info

    scores = root / "resource" / "scores.json"
    if not scores.is_file():
        scores = root / "scores.json"
    with open(scores, encoding="utf-8") as f:
        meta = json.load(f)
    info["scores"] = len(meta)

    if not (root / "WAVE").is_dir():
        issues.append("Missing WAVE/")

    with open(root / "train" / "wav.scp", encoding="utf-8") as f:
        uid, rel = f.readline().split(maxsplit=1)
        rel = rel.strip()
        wav = root / rel
        if uid not in meta:
            issues.append("scores.json mismatch wav.scp")
        elif not wav.is_file():
            issues.append(f"Missing audio: {wav}")

    return not issues, issues, info

import json
from pathlib import Path
from typing import Any, Dict, List, Optional

import torch
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import Dataset

DEFAULT_DATA_DIR = None

def _resolve_scores_path(root: Path) -> Path:
    for path in (root / "scores.json", root / "resource" / "scores.json"):
        if path.exists():
            return path
    raise FileNotFoundError(f"Missing scores.json under {root}")

def load_speechocean762(split: str = "train", data_dir: Optional[str] = None) -> List[Dict[str, Any]]:
    root = Path(data_dir or DEFAULT_DATA_DIR)
    if not root.exists():
        raise FileNotFoundError(f"Dataset not found: {root.resolve()}")

    with open(_resolve_scores_path(root), encoding="utf-8") as f:
        scores = json.load(f)

    wav_scp = root / split / "wav.scp"
    if not wav_scp.exists():
        raise FileNotFoundError(f"Missing {wav_scp}")

    samples = []
    with open(wav_scp, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            uid, path = line.split(maxsplit=1)
            if uid not in scores:
                continue
            wav_path = Path(path)
            if not wav_path.is_absolute():
                wav_path = (root / path).resolve()
            meta = scores[uid]
            samples.append(
                {
                    "id": uid,
                    "audio": str(wav_path),
                    "text": meta["text"],
                    "accuracy": meta.get("accuracy"),
                    "fluency": meta.get("fluency"),
                    "completeness": meta.get("completeness"),
                    "prosodic": meta.get("prosodic"),
                    "total": meta.get("total"),
                    "words": meta.get("words", []),
                }
            )
    return samples

class SpeechOcean762Dataset(Dataset):
    UTTERANCE_KEYS = ["accuracy", "fluency", "completeness", "prosodic", "total"]
    WORD_KEYS = ["accuracy", "stress", "total"]

    def __init__(
        self,
        split: str = "train",
        sample_rate: int = 16000,
        max_duration_sec: float = 15.0,
        score_scale: float = 5.0,
        data_dir: Optional[str] = None,
    ):
        self.sample_rate = sample_rate
        self.max_samples = int(max_duration_sec * sample_rate)
        self.score_scale = score_scale
        self.samples = load_speechocean762(split, data_dir)

    def __len__(self) -> int:
        return len(self.samples)

    def _load_audio(self, wav_path: str) -> torch.Tensor:
        import torchaudio

        wav, sr = torchaudio.load(wav_path)
        wav = wav.mean(dim=0)
        if sr != self.sample_rate:
            wav = torchaudio.functional.resample(wav, sr, self.sample_rate)
        if wav.shape[0] > self.max_samples:
            wav = wav[: self.max_samples]
        return wav

    def __getitem__(self, idx: int) -> Dict[str, Any]:
        item = self.samples[idx]
        words = item.get("words", [])
        phoneme_labels, word_labels = [], []
        phoneme_tokens, word_texts, word_phone_ranges = [], [], []

        for w in words:
            phones = w.get("phones", [])
            if isinstance(phones, str):
                phones = phones.split()
            pa = w.get("phones-accuracy", w.get("phones_accuracy", []))
            start = len(phoneme_tokens)
            phoneme_tokens.extend(phones)
            phoneme_labels.extend([float(x) for x in pa])
            word_phone_ranges.append((start, len(phoneme_tokens)))
            word_texts.append(w.get("text", ""))
            word_labels.append({k: float(w.get(k, 0)) / self.score_scale for k in self.WORD_KEYS})

        utterance_labels = {
            k: float(item.get(k, 0)) / self.score_scale
            for k in self.UTTERANCE_KEYS
            if item.get(k) is not None
        }
        if "completeness" not in utterance_labels:
            utterance_labels["completeness"] = utterance_labels.get("fluency", 0.0)

        return {
            "id": item["id"],
            "waveform": self._load_audio(item["audio"]),
            "text": item["text"],
            "phoneme_tokens": phoneme_tokens,
            "phoneme_labels": phoneme_labels,
            "word_labels": word_labels,
            "word_texts": word_texts,
            "word_phone_ranges": word_phone_ranges,
            "utterance_labels": utterance_labels,
        }

def collate_fn(batch: List[Dict[str, Any]]) -> Dict[str, Any]:
    waveforms = [b["waveform"] for b in batch]
    wav_lengths = torch.tensor([w.shape[0] for w in waveforms], dtype=torch.long)
    return {
        "ids": [b["id"] for b in batch],
        "waveforms": pad_sequence(waveforms, batch_first=True),
        "wav_lengths": wav_lengths,
        "texts": [b["text"] for b in batch],
        "phoneme_tokens": [b["phoneme_tokens"] for b in batch],
        "phoneme_labels": [b["phoneme_labels"] for b in batch],
        "word_labels": [b["word_labels"] for b in batch],
        "word_texts": [b["word_texts"] for b in batch],
        "word_phone_ranges": [b["word_phone_ranges"] for b in batch],
        "utterance_labels": [b["utterance_labels"] for b in batch],
    }

from contextlib import nullcontext
from typing import Optional

import torch
import torch.nn as nn
from transformers import WavLMModel

class WavLMEncoder(nn.Module):
    """
    Wraps `microsoft/wavlm-large` for pronunciation feature extraction.

    Output: frame-level representations at ~20ms stride (50 Hz for 16kHz audio
    with conv subsampling factor 320: 16000/320 = 50 frames/sec).
    """

    def __init__(
        self,
        model_name: str = "microsoft/wavlm-large",
        freeze: bool = True,
        use_lora: bool = False,
        lora_r: int = 8,
        lora_alpha: int = 16,
        lora_dropout: float = 0.05,
        lora_target_modules: Optional[list] = None,
    ):
        super().__init__()
        self.wavlm = WavLMModel.from_pretrained(model_name)
        self.output_dim = self.wavlm.config.hidden_size  # 1024 for large

        if freeze and not use_lora:
            for p in self.wavlm.parameters():
                p.requires_grad = False
            self.wavlm.eval()

        self._frozen = freeze and not use_lora

        if use_lora:
            from peft import LoraConfig, get_peft_model

            target = lora_target_modules or ["q_proj", "v_proj"]
            lora_config = LoraConfig(
                r=lora_r,
                lora_alpha=lora_alpha,
                target_modules=target,
                lora_dropout=lora_dropout,
                bias="none",
            )
            self.wavlm = get_peft_model(self.wavlm, lora_config)

    def forward(
        self,
        waveform: torch.Tensor,
        wav_lengths: Optional[torch.Tensor] = None,
        attention_mask: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        """
        Args:
            waveform: (B, num_samples) float tensor, 16kHz mono.
            wav_lengths: (B,) actual sample counts before padding.
            attention_mask: optional (B, num_samples), 1=valid 0=pad.

        Returns:
            hidden_states: (B, T_frames, D) last hidden layer output.
        """
        if attention_mask is None:
            B, S = waveform.shape
            if wav_lengths is not None:
                attention_mask = (
                    torch.arange(S, device=waveform.device).unsqueeze(0)
                    < wav_lengths.unsqueeze(1)
                ).long()
            else:
                # padded batch: trailing zeros from pad_sequence
                attention_mask = (waveform.abs() > 1e-8).long()

        if self._frozen:
            self.wavlm.eval()

        ctx = torch.inference_mode if self._frozen else nullcontext
        with ctx():
            outputs = self.wavlm(
                input_values=waveform,
                attention_mask=attention_mask,
            )
        return outputs.last_hidden_state

    def frame_lengths_from_samples(self, wav_lengths: torch.Tensor) -> torch.Tensor:
        """Exact WavLM frame counts from raw sample lengths (not samples//320)."""
        return self.wavlm._get_feat_extract_output_lengths(wav_lengths).long()

    def frame_rate(self, sample_rate: int = 16000) -> float:
        """Approximate frame rate (Hz) of WavLM output."""
        # WavLM conv feature extractor: total stride 320 samples
        return sample_rate / 320.0

import math
from typing import Optional

import torch
import torch.nn as nn

class TaskTransformerEncoder(nn.Module):
    """
    Additional Transformer encoder layers (2-4) on top of WavLM hidden states.

    Uses pre-norm TransformerEncoderLayer for training stability.
    """

    def __init__(
        self,
        input_dim: int,
        num_layers: int = 3,
        num_heads: int = 8,
        ff_dim: int = 4096,
        dropout: float = 0.1,
        max_seq_len: int = 2000,
    ):
        super().__init__()
        if input_dim % num_heads != 0:
            raise ValueError(
                f"input_dim ({input_dim}) must be divisible by num_heads ({num_heads})"
            )
        self.input_proj = nn.Linear(input_dim, input_dim)
        self.pos_encoding = SinusoidalPositionalEncoding(input_dim, max_seq_len)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=input_dim,
            nhead=num_heads,
            dim_feedforward=ff_dim,
            dropout=dropout,
            batch_first=True,
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.output_dim = input_dim

    def forward(
        self,
        x: torch.Tensor,
        src_key_padding_mask: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        """
        Args:
            x: (B, T, D) frame features from WavLM.
            src_key_padding_mask: (B, T) True = padded frame (ignore).

        Returns:
            (B, T, D) refined frame features.
        """
        x = self.input_proj(x)
        x = self.pos_encoding(x)
        return self.encoder(x, src_key_padding_mask=src_key_padding_mask)

class SinusoidalPositionalEncoding(nn.Module):
    """Standard sinusoidal position encoding added to frame features."""

    def __init__(self, dim: int, max_len: int = 2000):
        super().__init__()
        pe = torch.zeros(max_len, dim)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, dim, 2, dtype=torch.float) * (-math.log(10000.0) / dim)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0), persistent=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x + self.pe[:, : x.size(1), :]

from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F

# Lazy import: torchaudio may fail on some Windows/Python combos
torchaudio = None
forced_align = None

def _ensure_torchaudio():
    global torchaudio, forced_align
    if torchaudio is not None:
        return True
    try:
        import torchaudio as _ta
        from torchaudio.functional import forced_align as _fa

        torchaudio = _ta
        forced_align = _fa
        return True
    except (ImportError, OSError):
        return False

# espeak-style phoneme set subset (ARPAbet compatible with SpeechOcean762)
DEFAULT_PHONEME_VOCAB = [
    "<pad>", "<unk>", "|",  # blank, unknown, word boundary
    "AA0", "AA1", "AA2", "AE0", "AE1", "AE2", "AH0", "AH1", "AH2",
    "AO0", "AO1", "AO2", "AW0", "AW1", "AW2", "AY0", "AY1", "AY2",
    "B", "CH", "D", "DH", "EH0", "EH1", "EH2", "ER0", "ER1", "ER2",
    "EY0", "EY1", "EY2", "F", "G", "HH", "IH0", "IH1", "IH2",
    "IY0", "IY1", "IY2", "JH", "K", "L", "M", "N", "NG",
    "OW0", "OW1", "OW2", "OY0", "OY1", "OY2", "P", "R", "S", "SH",
    "T", "TH", "UH0", "UH1", "UH2", "UW0", "UW1", "UW2",
    "V", "W", "Y", "Z", "ZH",
]

@dataclass
class PhonemeAlignment:
    """Single phoneme alignment result."""

    phoneme: str
    token_id: int
    start_frame: int
    end_frame: int
    confidence: float

class CTCAligner(nn.Module):
    """
    CTC head + forced alignment to map phoneme sequence -> frame spans.

    Can optionally use a pretrained torchaudio CTC bundle for alignment-only
    mode; default trains a lightweight linear CTC head on WavLM features.
    """

    def __init__(
        self,
        input_dim: int,
        phoneme_vocab: Optional[List[str]] = None,
        use_pretrained_bundle: bool = False,
        blank_id: int = 0,
    ):
        super().__init__()
        self.phoneme_vocab = phoneme_vocab or DEFAULT_PHONEME_VOCAB
        self.token2id = {p: i for i, p in enumerate(self.phoneme_vocab)}
        self.blank_id = blank_id
        self.num_tokens = len(self.phoneme_vocab)

        self.ctc_proj = nn.Linear(input_dim, self.num_tokens)
        self.use_pretrained_bundle = use_pretrained_bundle
        self._bundle = None

        if use_pretrained_bundle and _ensure_torchaudio():
            try:
                # Phoneme ASR model (espeak phonemes) when available
                self._bundle = torchaudio.pipelines.MMS_FA
            except AttributeError:
                self._bundle = None

    def phonemes_to_ids(self, phonemes: List[str]) -> torch.Tensor:
        """Map ARPAbet phoneme strings to token IDs."""
        ids = []
        for p in phonemes:
            ids.append(self.token2id.get(p, self.token2id["<unk>"]))
        return torch.tensor(ids, dtype=torch.long)

    def forward_ctc_logits(self, frame_features: torch.Tensor) -> torch.Tensor:
        """(B, T, D) -> (T, B, C) log-probs for torch.nn.functional.ctc_loss."""
        logits = self.ctc_proj(frame_features)
        log_probs = F.log_softmax(logits, dim=-1)
        return log_probs.transpose(0, 1)

    def _align_log_probs(self, frame_features: torch.Tensor) -> torch.Tensor:
        """(T, D) -> (1, T, C) log-probs for torchaudio forced_align (batch-first)."""
        logits = self.ctc_proj(frame_features.unsqueeze(0))
        return F.log_softmax(logits, dim=-1)

    def _run_forced_align(
        self,
        log_probs: torch.Tensor,
        targets: torch.Tensor,
        input_lengths: torch.Tensor,
        target_lengths: torch.Tensor,
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """forced_align wants (B, T, C) with B=1; GPU kernel errors on wrong layout."""
        assert log_probs.dim() == 3 and log_probs.shape[0] == 1
        try:
            return forced_align(
                log_probs.float(),
                targets.unsqueeze(0),
                input_lengths,
                target_lengths,
                blank=self.blank_id,
            )
        except RuntimeError:
            return forced_align(
                log_probs.float().cpu(),
                targets.unsqueeze(0).cpu(),
                input_lengths.cpu(),
                target_lengths.cpu(),
                blank=self.blank_id,
            )

    def align_utterance(
        self,
        frame_features: torch.Tensor,
        phonemes: List[str],
        frame_lengths: Optional[int] = None,
    ) -> Tuple[List[PhonemeAlignment], torch.Tensor]:
        """
        Force-align one utterance.

        Args:
            frame_features: (T, D) single utterance frame features.
            phonemes: canonical ARPAbet phoneme list from transcript/CMUdict.
            frame_lengths: number of valid frames T.

        Returns:
            alignments: list of PhonemeAlignment with frame spans.
            node_features: (num_phonemes, D) pooled features for GAT nodes.
        """
        if frame_lengths is None:
            frame_lengths = frame_features.shape[0]

        T, D = frame_features.shape
        if len(phonemes) == 0:
            return [], frame_features.new_zeros(0, D)

        targets = self.phonemes_to_ids(phonemes).to(frame_features.device)

        if not _ensure_torchaudio() or forced_align is None:
            return self._uniform_align(frame_features, phonemes)

        log_probs = self._align_log_probs(frame_features)  # (1, T, C)
        input_lengths = torch.tensor([frame_lengths], device=frame_features.device)
        target_lengths = torch.tensor([len(targets)], device=frame_features.device)

        aligned_tokens, _align_scores = self._run_forced_align(
            log_probs, targets, input_lengths, target_lengths
        )
        if aligned_tokens.dim() > 1:
            aligned = aligned_tokens[0, :frame_lengths]
        else:
            aligned = aligned_tokens[:frame_lengths]

        spans = self._tokens_to_spans(aligned, targets, phonemes)
        node_features = self._pool_node_features(frame_features, spans)
        return spans, node_features

    def _tokens_to_spans(
        self,
        aligned: torch.Tensor,
        targets: torch.Tensor,
        phonemes: List[str],
    ) -> List[PhonemeAlignment]:
        """
        Parse per-frame CTC alignment into phoneme start/end spans.

        forced_align output assigns each frame to a target token index or blank.
        Consecutive frames with the same non-blank token index form one span.
        """
        spans: List[PhonemeAlignment] = []
        target_list = targets.tolist()
        i = 0
        while i < len(phonemes):
            token_id = target_list[i]
            # find frames assigned to this target position
            mask = aligned == i  # forced_align uses target position indices
            if mask.any():
                idx = mask.nonzero(as_tuple=True)[0]
                start_f, end_f = int(idx[0]), int(idx[-1])
                conf = float(mask.float().mean())
            else:
                # phoneme got no frames — interpolate
                start_f = end_f = 0
                conf = 0.0
            spans.append(
                PhonemeAlignment(
                    phoneme=phonemes[i],
                    token_id=token_id,
                    start_frame=start_f,
                    end_frame=end_f,
                    confidence=conf,
                )
            )
            i += 1
        return spans

    def _pool_node_features(
        self,
        frame_features: torch.Tensor,
        spans: List[PhonemeAlignment],
    ) -> torch.Tensor:
        """
        Pool frame hidden states over [t_start, t_end] for each phoneme.

        This is the critical bridge from CTC alignment to Graph Attention:
        each phoneme node receives a fixed-size embedding regardless of
        how many frames it spans.
        """
        nodes = []
        T = frame_features.shape[0]
        for span in spans:
            s = max(0, span.start_frame)
            e = min(T - 1, span.end_frame)
            if s <= e:
                pooled = frame_features[s : e + 1].mean(dim=0)
            else:
                pooled = frame_features.mean(dim=0)
            nodes.append(pooled)
        return torch.stack(nodes, dim=0) if nodes else frame_features.new_zeros(0, frame_features.shape[-1])

    def _uniform_align(
        self,
        frame_features: torch.Tensor,
        phonemes: List[str],
    ) -> Tuple[List[PhonemeAlignment], torch.Tensor]:
        """Fallback equal-split alignment when forced_align is unavailable."""
        T, D = frame_features.shape
        n = max(len(phonemes), 1)
        chunk = T // n
        spans = []
        for i, ph in enumerate(phonemes):
            s = i * chunk
            e = min(T - 1, (i + 1) * chunk - 1) if i < n - 1 else T - 1
            tid = self.token2id.get(ph, self.token2id["<unk>"])
            spans.append(
                PhonemeAlignment(ph, tid, s, e, 1.0)
            )
        node_features = self._pool_node_features(frame_features, spans)
        return spans, node_features

    def batch_align(
        self,
        frame_features: torch.Tensor,
        phoneme_lists: List[List[str]],
        frame_lengths: torch.Tensor,
    ) -> List[Tuple[List[PhonemeAlignment], torch.Tensor]]:
        """Align a batch; returns per-utterance (spans, node_features)."""
        results = []
        B = frame_features.shape[0]
        for b in range(B):
            T_b = int(frame_lengths[b].item())
            spans, nodes = self.align_utterance(
                frame_features[b, :T_b],
                phoneme_lists[b],
                T_b,
            )
            results.append((spans, nodes))
        return results

    def ctc_loss(
        self,
        frame_features: torch.Tensor,
        phoneme_lists: List[List[str]],
        frame_lengths: torch.Tensor,
    ) -> torch.Tensor:
        """Auxiliary CTC loss for training the alignment head."""
        log_probs = self.forward_ctc_logits(frame_features)
        targets_list = []
        target_lengths = []
        for phs in phoneme_lists:
            t = self.phonemes_to_ids(phs)
            targets_list.append(t)
            target_lengths.append(len(t))
        targets = torch.cat(targets_list).to(frame_features.device)
        target_lengths_t = torch.tensor(target_lengths, device=frame_features.device)
        loss = F.ctc_loss(
            log_probs,
            targets,
            frame_lengths,
            target_lengths_t,
            blank=self.blank_id,
            zero_infinity=True,
        )
        return loss

from typing import List, Optional, Tuple

import torch
import torch.nn as nn

try:
    from torch_geometric.nn import GATv2Conv
except ImportError:
    GATv2Conv = None

class PhonemeGraphNetwork(nn.Module):
    """GATv2-based phoneme graph encoder."""

    def __init__(
        self,
        input_dim: int,
        hidden_dim: int = 256,
        num_layers: int = 2,
        num_heads: int = 4,
        dropout: float = 0.1,
        edge_sequential: bool = True,
        edge_same_word: bool = True,
        edge_same_syllable: bool = False,
    ):
        super().__init__()
        if GATv2Conv is None:
            raise ImportError("torch_geometric is required for PhonemeGraphNetwork")

        self.input_proj = nn.Linear(input_dim, hidden_dim)
        self.edge_sequential = edge_sequential
        self.edge_same_word = edge_same_word
        self.edge_same_syllable = edge_same_syllable

        self.gat_layers = nn.ModuleList()
        for i in range(num_layers):
            in_ch = hidden_dim
            out_ch = hidden_dim // num_heads
            self.gat_layers.append(
                GATv2Conv(
                    in_channels=in_ch,
                    out_channels=out_ch,
                    heads=num_heads,
                    dropout=dropout,
                    concat=True,
                )
            )
        self.norm = nn.LayerNorm(hidden_dim)
        self.dropout = nn.Dropout(dropout)
        self.output_dim = hidden_dim

    @staticmethod
    def build_edge_index(
        num_phonemes: int,
        word_phone_ranges: List[Tuple[int, int]],
        sequential: bool = True,
        same_word: bool = True,
        same_syllable: bool = False,
    ) -> torch.Tensor:
        """
        Build COO edge_index (2, E) for one utterance.

        Args:
            num_phonemes: total phoneme count.
            word_phone_ranges: list of (start, end) exclusive indices per word.
        """
        edges = set()

        if sequential:
            for i in range(num_phonemes - 1):
                edges.add((i, i + 1))
                edges.add((i + 1, i))

        if same_word:
            for start, end in word_phone_ranges:
                for i in range(start, end):
                    for j in range(start, end):
                        if i != j:
                            edges.add((i, j))

        if same_syllable:
            # simple heuristic: split each word's phones in half
            for start, end in word_phone_ranges:
                mid = (start + end) // 2
                for i in range(start, mid):
                    for j in range(start, mid):
                        if i != j:
                            edges.add((i, j))
                for i in range(mid, end):
                    for j in range(mid, end):
                        if i != j:
                            edges.add((i, j))

        if not edges:
            # self-loop fallback for single phoneme
            edges.add((0, 0))

        src, dst = zip(*edges)
        return torch.tensor([src, dst], dtype=torch.long)

    def forward_single(
        self,
        node_features: torch.Tensor,
        word_phone_ranges: List[Tuple[int, int]],
    ) -> torch.Tensor:
        """
        Args:
            node_features: (N, D) pooled phoneme embeddings from CTC align step.
            word_phone_ranges: phoneme index ranges per word.

        Returns:
            (N, hidden_dim) context-enriched phoneme embeddings.
        """
        x = self.input_proj(node_features)
        edge_index = self.build_edge_index(
            node_features.shape[0],
            word_phone_ranges,
            self.edge_sequential,
            self.edge_same_word,
            self.edge_same_syllable,
        ).to(node_features.device)

        for gat in self.gat_layers:
            x = gat(x, edge_index)
            x = self.dropout(torch.relu(x))
        return self.norm(x)

    def forward_batch(
        self,
        node_features_list: List[torch.Tensor],
        word_phone_ranges_list: List[List[Tuple[int, int]]],
    ) -> List[torch.Tensor]:
        """Process variable-size graphs per utterance."""
        outputs = []
        for nodes, ranges in zip(node_features_list, word_phone_ranges_list):
            if nodes.shape[0] == 0:
                outputs.append(nodes)
            else:
                outputs.append(self.forward_single(nodes, ranges))
        return outputs

from typing import Dict, List, Optional, Tuple

import torch
import torch.nn as nn

class RegressionHead(nn.Module):
    """Single-aspect regression head with layer norm (GOPT style)."""

    def __init__(self, input_dim: int, dropout: float = 0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.LayerNorm(input_dim),
            nn.Dropout(dropout),
            nn.Linear(input_dim, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x).squeeze(-1)

class MultiTaskHeads(nn.Module):
    """Collection of aspect-specific heads at phoneme/word/utterance levels."""

    def __init__(
        self,
        input_dim: int,
        utterance_aspects: Optional[List[str]] = None,
        word_aspects: Optional[List[str]] = None,
        phoneme_aspects: Optional[List[str]] = None,
        dropout: float = 0.1,
    ):
        super().__init__()
        self.utterance_aspects = utterance_aspects or [
            "accuracy", "fluency", "completeness", "prosodic", "total"
        ]
        self.word_aspects = word_aspects or ["accuracy", "stress", "total"]
        self.phoneme_aspects = phoneme_aspects or ["accuracy"]

        self.utt_heads = nn.ModuleDict(
            {a: RegressionHead(input_dim, dropout) for a in self.utterance_aspects}
        )
        self.word_heads = nn.ModuleDict(
            {a: RegressionHead(input_dim, dropout) for a in self.word_aspects}
        )
        self.phoneme_heads = nn.ModuleDict(
            {a: RegressionHead(input_dim, dropout) for a in self.phoneme_aspects}
        )

    def pool_utterance(self, phoneme_embeddings: torch.Tensor) -> torch.Tensor:
        """Mean-pool all phoneme nodes -> utterance representation."""
        if phoneme_embeddings.shape[0] == 0:
            return phoneme_embeddings.new_zeros(phoneme_embeddings.shape[-1])
        return phoneme_embeddings.mean(dim=0)

    def pool_words(
        self,
        phoneme_embeddings: torch.Tensor,
        word_phone_ranges: List[Tuple[int, int]],
    ) -> torch.Tensor:
        """Mean-pool phoneme nodes per word -> (num_words, D)."""
        word_embs = []
        for start, end in word_phone_ranges:
            if start < end:
                word_embs.append(phoneme_embeddings[start:end].mean(dim=0))
            else:
                word_embs.append(phoneme_embeddings.new_zeros(phoneme_embeddings.shape[-1]))
        return torch.stack(word_embs, dim=0) if word_embs else phoneme_embeddings.new_zeros(0, phoneme_embeddings.shape[-1])

    def forward_single(
        self,
        phoneme_embeddings: torch.Tensor,
        word_phone_ranges: List[Tuple[int, int]],
    ) -> Dict[str, torch.Tensor]:
        """
        Predict all aspects for one utterance.

        Returns dict with keys:
          utterance_{aspect}, word_{aspect} (W,), phoneme_{aspect} (N,)
        """
        utt_repr = self.pool_utterance(phoneme_embeddings)
        word_repr = self.pool_words(phoneme_embeddings, word_phone_ranges)

        out: Dict[str, torch.Tensor] = {}
        for aspect, head in self.utt_heads.items():
            out[f"utterance_{aspect}"] = head(utt_repr.unsqueeze(0)).squeeze(0)
        for aspect, head in self.word_heads.items():
            if word_repr.shape[0] > 0:
                out[f"word_{aspect}"] = head(word_repr)
            else:
                out[f"word_{aspect}"] = phoneme_embeddings.new_zeros(0)
        for aspect, head in self.phoneme_heads.items():
            if phoneme_embeddings.shape[0] > 0:
                out[f"phoneme_{aspect}"] = head(phoneme_embeddings)
            else:
                out[f"phoneme_{aspect}"] = phoneme_embeddings.new_zeros(0)
        return out

    def forward_batch(
        self,
        phoneme_embeddings_list: List[torch.Tensor],
        word_phone_ranges_list: List[List[Tuple[int, int]]],
    ) -> List[Dict[str, torch.Tensor]]:
        return [
            self.forward_single(pe, wr)
            for pe, wr in zip(phoneme_embeddings_list, word_phone_ranges_list)
        ]

from typing import Any, Dict, List, Optional

import torch
import torch.nn as nn

class PronunciationAssessmentModel(nn.Module):
    """Full pronunciation assessment pipeline."""

    def __init__(self, config: Dict[str, Any]):
        super().__init__()
        wavlm_cfg = config.get("wavlm", {})
        trans_cfg = config.get("transformer", {})
        ctc_cfg = config.get("ctc_align", {})
        graph_cfg = config.get("phoneme_graph", {})
        mt_cfg = config.get("multitask", {})

        self.wavlm = WavLMEncoder(
            model_name=wavlm_cfg.get("model_name", "microsoft/wavlm-large"),
            freeze=wavlm_cfg.get("freeze", True),
            use_lora=wavlm_cfg.get("use_lora", False),
            lora_r=wavlm_cfg.get("lora_r", 8),
            lora_alpha=wavlm_cfg.get("lora_alpha", 16),
            lora_dropout=wavlm_cfg.get("lora_dropout", 0.05),
            lora_target_modules=wavlm_cfg.get("lora_target_modules"),
        )
        d = self.wavlm.output_dim

        self.task_transformer = TaskTransformerEncoder(
            input_dim=d,
            num_layers=trans_cfg.get("num_layers", 3),
            num_heads=trans_cfg.get("num_heads", 8),
            ff_dim=trans_cfg.get("ff_dim", 3072),
            dropout=trans_cfg.get("dropout", 0.1),
            max_seq_len=trans_cfg.get("max_seq_len", 2000),
        )

        self.ctc_aligner = CTCAligner(
            input_dim=d,
            use_pretrained_bundle=not ctc_cfg.get("use_wavlm_ctc_head", True),
        )

        graph_hidden = graph_cfg.get("hidden_dim", 256)
        self.phoneme_graph = PhonemeGraphNetwork(
            input_dim=d,
            hidden_dim=graph_hidden,
            num_layers=graph_cfg.get("num_gat_layers", 2),
            num_heads=graph_cfg.get("num_heads", 4),
            dropout=graph_cfg.get("dropout", 0.1),
            edge_sequential=graph_cfg.get("edge_types", {}).get("sequential", True),
            edge_same_word=graph_cfg.get("edge_types", {}).get("same_word", True),
            edge_same_syllable=graph_cfg.get("edge_types", {}).get("same_syllable", False),
        )

        self.multitask_heads = MultiTaskHeads(
            input_dim=graph_hidden,
            utterance_aspects=mt_cfg.get("utterance_aspects"),
            word_aspects=mt_cfg.get("word_aspects"),
            phoneme_aspects=mt_cfg.get("phoneme_aspects"),
            dropout=mt_cfg.get("dropout", 0.1),
        )

        self.sample_rate = config.get("train", {}).get("dataset", {}).get("sample_rate", 16000)

    def _frame_lengths_from_wave(self, wav_lengths: torch.Tensor) -> torch.Tensor:
        """Convert sample lengths to WavLM frame lengths using model conv math."""
        return self.wavlm.frame_lengths_from_samples(wav_lengths)

    def forward(
        self,
        waveforms: torch.Tensor,
        wav_lengths: torch.Tensor,
        phoneme_tokens: List[List[str]],
        word_phone_ranges: List[List[tuple]],
        return_alignments: bool = False,
    ) -> Dict[str, Any]:
        """
        Forward pass for a batch.

        Args:
            waveforms: (B, samples) padded.
            wav_lengths: (B,) actual sample counts.
            phoneme_tokens: list of phoneme strings per utterance.
            word_phone_ranges: word -> phoneme index ranges.

        Returns:
            dict with per-utterance predictions, optional alignments, ctc_loss.
        """
        # Step 1: WavLM features
        frame_feats = self.wavlm(waveforms, wav_lengths=wav_lengths)  # (B, T, D)
        frame_lengths = self._frame_lengths_from_wave(wav_lengths)

        # padding mask for transformer
        T = frame_feats.shape[1]
        pad_mask = torch.arange(T, device=waveforms.device).unsqueeze(0) >= frame_lengths.unsqueeze(1)

        # Step 2: task transformer
        frame_feats = self.task_transformer(frame_feats, src_key_padding_mask=pad_mask)

        # Step 3: CTC alignment -> phoneme node features
        align_results = self.ctc_aligner.batch_align(
            frame_feats, phoneme_tokens, frame_lengths
        )
        node_features_list = [nodes for _, nodes in align_results]
        alignments_list = [spans for spans, _ in align_results] if return_alignments else None

        # Step 4: graph attention
        graph_out = self.phoneme_graph.forward_batch(
            node_features_list, word_phone_ranges
        )

        # Step 5: multi-task heads
        predictions = self.multitask_heads.forward_batch(graph_out, word_phone_ranges)

        # auxiliary CTC loss
        ctc_loss = self.ctc_aligner.ctc_loss(frame_feats, phoneme_tokens, frame_lengths)

        out = {
            "predictions": predictions,
            "ctc_loss": ctc_loss,
            "graph_embeddings": graph_out,
        }
        if return_alignments:
            out["alignments"] = alignments_list
        return out

    @classmethod
    def from_config_path(cls, path: str) -> "PronunciationAssessmentModel":
        import yaml

        with open(path, encoding="utf-8") as f:
            config = yaml.safe_load(f)
        return cls(config)

#!/usr/bin/env python3
"""Train pronunciation model. Usage: python train.py [--config config.yaml]"""

from contextlib import nullcontext
import logging
import time
from pathlib import Path
from typing import Any, Dict, List, Optional

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from tqdm import tqdm

logger = logging.getLogger(__name__)

class MultiTaskLoss(nn.Module):
  UTTERANCE_KEYS = ["accuracy", "fluency", "completeness", "prosodic", "total"]
  WORD_KEYS = ["accuracy", "stress", "total"]

  def __init__(self, weights: Optional[Dict[str, float]] = None, ctc_weight: float = 0.1):
    super().__init__()
    self.weights = weights or {"utterance": 1.0, "word": 1.0, "phoneme": 1.0}
    self.ctc_weight = ctc_weight
    self.mse = nn.MSELoss()

  def forward(self, predictions, utterance_labels, word_labels, phoneme_labels, ctc_loss):
    device = ctc_loss.device
    utt_l, word_l, ph_l = [], [], []
    for b, pred in enumerate(predictions):
      for key in self.UTTERANCE_KEYS:
        pkey = f"utterance_{key}"
        if pkey in pred and key in utterance_labels[b]:
          utt_l.append(self.mse(pred[pkey], torch.tensor(utterance_labels[b][key], device=device)))
      for key in self.WORD_KEYS:
        pkey = f"word_{key}"
        if pkey in pred and pred[pkey].numel() > 0:
          n = pred[pkey].shape[0]
          t = torch.tensor([word_labels[b][i].get(key, 0.0) for i in range(n)], device=device)
          word_l.append(self.mse(pred[pkey], t))
      pkey = "phoneme_accuracy"
      if pkey in pred and pred[pkey].numel() > 0:
        n = pred[pkey].shape[0]
        t = torch.tensor(phoneme_labels[b][:n], device=device, dtype=torch.float32)
        ph_l.append(self.mse(pred[pkey], t))

    def _mean(xs):
      return torch.stack(xs).mean() if xs else torch.tensor(0.0, device=device)

    l_utt, l_word, l_ph = _mean(utt_l), _mean(word_l), _mean(ph_l)
    total = self.weights["utterance"] * l_utt + self.weights["word"] * l_word + self.weights["phoneme"] * l_ph + self.ctc_weight * ctc_loss
    return {"loss": total, "utterance_loss": l_utt, "word_loss": l_word, "phoneme_loss": l_ph, "ctc_loss": ctc_loss}

class Trainer:
  def __init__(self, config: Dict[str, Any]):
    self.config = config
    tc = config["train"]
    self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    self.use_amp = bool(tc.get("use_amp", False)) and self.device.type == "cuda"
    self.scaler = torch.cuda.amp.GradScaler() if self.use_amp else None
    norm = 10.0 / config["multitask"]["score_scale"]
    ds = tc["dataset"]
    kw = dict(sample_rate=ds["sample_rate"], max_duration_sec=ds["max_duration_sec"], score_scale=norm, data_dir=ds.get("data_dir"))
    self.train_ds = SpeechOcean762Dataset(ds.get("train_split", "train"), **kw)
    self.val_ds = SpeechOcean762Dataset(ds.get("test_split", "test"), **kw)
    pin = bool(tc.get("pin_memory", False)) and self.device.type == "cuda"
    self.train_loader = DataLoader(self.train_ds, tc["batch_size"], shuffle=True, num_workers=tc.get("num_workers", 2), collate_fn=collate_fn, pin_memory=pin)
    self.val_loader = DataLoader(self.val_ds, tc["batch_size"], shuffle=False, num_workers=tc.get("num_workers", 2), collate_fn=collate_fn, pin_memory=pin)
    self.model = PronunciationAssessmentModel(config).to(self.device)
    self.criterion = MultiTaskLoss(tc.get("loss_weights"))
    self.optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, self.model.parameters()), lr=tc["learning_rate"], weight_decay=tc.get("weight_decay", 0.01))
    self.num_epochs = tc["num_epochs"]
    self.grad_clip = tc.get("grad_clip", 1.0)
    self.grad_accum = max(1, int(tc.get("grad_accum_steps", 1)))
    self.save_every = tc.get("save_every_epochs", 5)
    self.ckpt_dir = Path(config["paths"]["checkpoint_dir"])
    self.ckpt_dir.mkdir(parents=True, exist_ok=True)

  def _step(self, batch, train=True, accum_scale=1.0):
    waveforms = batch["waveforms"].to(self.device)
    wav_lengths = batch["wav_lengths"].to(self.device)
    ctx = torch.cuda.amp.autocast if train and self.use_amp else nullcontext
    with ctx():
      out = self.model(waveforms, wav_lengths, batch["phoneme_tokens"], batch["word_phone_ranges"])
      ld = self.criterion(out["predictions"], batch["utterance_labels"], batch["word_labels"], batch["phoneme_labels"], out["ctc_loss"])
      loss = ld["loss"] / accum_scale
    if train:
      if self.use_amp:
        self.scaler.scale(loss).backward()
      else:
        loss.backward()
    ld = dict(ld)
    ld["loss"] = loss.detach() * accum_scale
    return ld

  def _optimizer_step(self):
    if self.use_amp:
      if self.grad_clip:
        self.scaler.unscale_(self.optimizer)
        torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.grad_clip)
      self.scaler.step(self.optimizer)
      self.scaler.update()
    else:
      if self.grad_clip:
        torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.grad_clip)
      self.optimizer.step()
    self.optimizer.zero_grad()

  def train(self):
    logging.basicConfig(level=logging.INFO)
    logger.info("Device=%s AMP=%s train=%d", self.device, self.use_amp, len(self.train_ds))
    best = float("inf")
    for epoch in range(1, self.num_epochs + 1):
      t0 = time.time()
      self.model.train()
      tr_loss, n = 0.0, 0
      self.optimizer.zero_grad()
      for i, batch in enumerate(tqdm(self.train_loader, desc=f"Epoch {epoch}")):
        tr_loss += float(self._step(batch, True, self.grad_accum)["loss"].cpu())
        n += 1
        if i % self.grad_accum == self.grad_accum - 1 or i == len(self.train_loader) - 1:
          self._optimizer_step()
        if self.device.type == "cuda":
          torch.cuda.empty_cache()
      self.model.eval()
      va_loss, m = 0.0, 0
      with torch.no_grad():
        for batch in tqdm(self.val_loader, desc="Val"):
          va_loss += float(self._step(batch, False)["loss"].cpu())
          m += 1
      tr, va = tr_loss / max(n, 1), va_loss / max(m, 1)
      logger.info("Epoch %d (%.0fs) train=%.4f val=%.4f", epoch, time.time() - t0, tr, va)
      if va < best:
        best = va
        torch.save(self.model.state_dict(), self.ckpt_dir / "best_model.pt")
      if epoch % self.save_every == 0:
        torch.save({"epoch": epoch, "model_state_dict": self.model.state_dict(), "config": self.config}, self.ckpt_dir / f"epoch_{epoch}.pt")

In [3]:
import os, shutil, torch
from pathlib import Path

os.environ.setdefault("PYTORCH_ALLOC_CONF", "expandable_segments:True")
os.makedirs("/kaggle/working/hf_cache", exist_ok=True)
os.environ["HF_HOME"] = os.environ["TRANSFORMERS_CACHE"] = "/kaggle/working/hf_cache"
assert torch.cuda.is_available(), "Settings → Accelerator → GPU T4"
torch.cuda.empty_cache()

data_dir = find_speechocean762_dir()
CONFIG["train"]["dataset"]["data_dir"] = str(data_dir)
ok, issues, info = verify_speechocean762(data_dir)
print("Dataset:", info)
if not ok:
    raise RuntimeError("\n".join(issues))

Trainer(CONFIG).train()
shutil.copy("/kaggle/working/checkpoints/best_model.pt", "/kaggle/working/best_model.pt")
print("Done → /kaggle/working/best_model.pt")

Dataset: {'root': '/kaggle/input/datasets/tnguynthnh142/speechocean762/speechocean762', 'scores': 5000}


/tmp/ipykernel_23/3280059715.py:1141: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler() if self.use_amp else None


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.26G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/488 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/1.26G [00:00<?, ?B/s]

/tmp/ipykernel_23/3280059715.py:368: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
INFO:__main__:Device=cuda AMP=True train=2500

Epoch 1:   0%|          | 0/1250 [00:00<?, ?it/s]/tmp/ipykernel_23/3280059715.py:1164: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with ctx():
/usr/local/lib/python3.12/dist-packages/torch/nn/functional.py:6371: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(

Val: 100%|██████████| 1250/1250 [06:33<00:00,  3.17it/s]
INFO:__main__:Epoch 1 (710s) train=1.0553 val=0.6076
Val: 100%|██████████| 1250/1250 [06:14<00:00,  3.34it/s]
INFO:__main__:Epoch 2 (664s) train=0.5815 val=0.4132
Val: 100%|██████████| 1250/1250 [06:14<00:00,  3.33i

Done → /kaggle/working/best_model.pt
